In [ ]:
import gymnasium as gym
import numpy as np
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.evaluation import evaluate_policy
import time
import os
import cv2
import glob
from datetime import datetime
from gymnasium.wrappers import RecordVideo

def make_atari_env(env_id, render_mode):
    """
    Create an Atari environment with proper wrappers.
    Ensures RGB observations instead of grayscale.
    """
    env = gym.make(env_id, render_mode=render_mode)
    env = AtariWrapper(env)
    return env

def preprocess_observation(obs):
    """
    Ensures the observation is in the correct format (3, 210, 160).
    """
    if obs.shape[-1] == 1:  # If grayscale (84, 84, 1)
        obs = np.repeat(obs, 3, axis=-1)  # Convert to (84, 84, 3)
    obs = cv2.resize(obs, (160, 210))  # Resize to (210, 160, 3)
    obs = np.transpose(obs, (2, 0, 1))  # Convert to (3, 210, 160)
    return obs

def record_video(env_name, model, num_episodes=10):
    """
    Records gameplay video of the trained agent.
    """
    # Create video directory
    video_dir = f"./videos/{env_name.split('/')[-1]}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(video_dir, exist_ok=True)

    # Create environment with RecordVideo wrapper
    env = make_atari_env(env_name, render_mode="rgb_array")
    env = RecordVideo(
        env,
        video_folder=video_dir,
        episode_trigger=lambda x: True,  # Record every episode
        name_prefix=f"{env_name.split('/')[-1]}"
    )

    rewards = []
    for episode in range(num_episodes):
        obs, info = env.reset()
        episode_reward = 0
        done, truncated = False, False

        while not (done or truncated):
            obs = preprocess_observation(obs)  # Ensure correct shape
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
            episode_reward += reward

        rewards.append(episode_reward)
        print(f"Episode {episode+1} reward: {episode_reward}")

    env.close()
    print(f"Videos saved to {video_dir}")
    return rewards, video_dir

def merge_videos(video_dir, output_filename="merged_video.mp4", fps=30):
    """
    Merges all videos in the given directory into a single video.
    """
    video_files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))
    if not video_files:
        print("No videos found to merge.")
        return None

    # Get video properties from the first video
    first_video = cv2.VideoCapture(video_files[0])
    frame_width = int(first_video.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(first_video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    output_path = os.path.join(video_dir, output_filename)

    # Initialize VideoWriter
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    for video_file in video_files:
        cap = cv2.VideoCapture(video_file)
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            out.write(frame)
        cap.release()

    out.release()
    print(f"Merged video saved to: {output_path}")
    return output_path

if __name__ == "__main__":
    # Load the trained model
    model_path = "dqn_model.zip"
    model = DQN.load(model_path, buffer_size=10000)
    print(f"Model loaded from {model_path}")

    # Environment setup
    env_name = "ALE/Breakout-v5"

    # Record video
    print("\n=== Recording gameplay videos ===")
    video_rewards, video_dir = record_video(env_name, model, num_episodes=100)

    # Merge recorded videos
    print("\n=== Merging recorded videos ===")
    merged_video_path = merge_videos(video_dir)
    if merged_video_path:
        print(f"Merged video available at: {merged_video_path}")

    # # Display gameplay in real-time
    # print("\n=== Displaying gameplay in real-time ===")
    # env = make_atari_env(env_name, render_mode="human")

    num_episodes = 5
    episode_rewards = []

    for episode in range(num_episodes):
        print(f"Starting episode {episode+1}/{num_episodes}")
        obs, info = env.reset()
        episode_reward = 0
        done, truncated = False, False

        while not (done or truncated):
            obs = preprocess_observation(obs)  # Ensure correct shape
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
            episode_reward += reward
            time.sleep(0.01)  # Delay for watchability

        episode_rewards.append(episode_reward)
        print(f"Episode {episode+1} reward: {episode_reward}")

    env.close()

    # Display performance summary
    print("\n=== Performance Summary ===")
    print(f"Average reward (display): {np.mean(episode_rewards):.2f}")
    print(f"Average reward (video): {np.mean(video_rewards):.2f}")
    print(f"Videos saved to: {video_dir}")
    if merged_video_path:
        print(f"Merged video located at: {merged_video_path}")

Model loaded from dqn_model.zip

=== Recording gameplay videos ===


c:\Users\ElvisGuy\.conda\envs\primary\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at e:\Work\ALU\ALU Machine Learning\Groups\Deep_Q_Learning\videos\Breakout-v5_20250321_190017 folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Episode 1 reward: 0.0
Episode 2 reward: 1.0
Episode 3 reward: 1.0
Episode 4 reward: 1.0
Episode 5 reward: 0.0
Episode 6 reward: 0.0
Episode 7 reward: 1.0
Episode 8 reward: 1.0
Episode 9 reward: 1.0
Episode 10 reward: 1.0
Episode 11 reward: 0.0
Episode 12 reward: 1.0
Episode 13 reward: 1.0
Episode 14 reward: 0.0
Episode 15 reward: 1.0
Episode 16 reward: 0.0
Episode 17 reward: 2.0
Episode 18 reward: 0.0
Episode 19 reward: 0.0
Episode 20 reward: 0.0
Episode 21 reward: 0.0
Episode 22 reward: 1.0
Episode 23 reward: 1.0
Episode 24 reward: 1.0
Episode 25 reward: 1.0
Episode 26 reward: 0.0
Episode 27 reward: 1.0
Episode 28 reward: 0.0
Episode 29 reward: 1.0
Episode 30 reward: 1.0
Episode 31 reward: 0.0
Episode 32 reward: 1.0
Episode 33 reward: 1.0
Episode 34 reward: 1.0
Episode 35 reward: 1.0
Episode 36 reward: 0.0
Episode 37 reward: 1.0
Episode 38 reward: 1.0
Episode 39 reward: 1.0
Episode 40 reward: 1.0
Episode 41 reward: 0.0
Episode 42 reward: 1.0
Episode 43 reward: 1.0
Episode 44 reward: 1

: 